# 1.5. Benchmark Hiệu năng Crawler M1 trên Google Colab CPU-only

## 1. Môi trường đo kiểm toán và đủ 7 chỉ số hiệu năng bắt buộc của Milestone 1

Toàn bộ các phiên thu thập của 5 thành viên (bao gồm cả phiên chạy gốc `4.425` bài và phiên `--resume` hoàn tất `5.000` bài của `TRI`) đều thực thi trên **Google Colab Standard CPU-only** (`is_google_colab = True`, `cpu_only = True`, `2 logical vCPU`, tổng RAM `12.975,52 MiB` ≈ `12,67 GiB`). Trình đo `ResourceProfiler` lấy mẫu cây tiến trình cha + con mỗi `0,5 giây` (đối chiếu chéo `ru_maxrss` từ nhân Linux) và quét dung lượng đĩa định kỳ mỗi `30 giây`.

```{note}
**Lưu ý chuẩn hóa đơn vị nhị phân `MiB`:**
Trong `src/crawler/profiler.py` (`BYTES_PER_MB = 1024**2 = 1.048.576`), các khóa JSON nội bộ mang hậu tố lịch sử `*_mb` nhưng đều được quy đổi chính xác theo cơ số nhị phân $1024^2$ byte (**`MiB`**). Mọi bảng và biểu đồ dưới đây trình bày thống nhất theo chuẩn **`MiB` / `GiB`**.
```

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
ps = m1["performance_summary"]
df_7m = pd.DataFrame([
    ("1. Thời gian xử lý 1 bài đơn lẻ (1-version, tuần tự từ lạnh / khấu hao theo lô)",
     f"{fmt_num(ps['single_paper_cold_1ver_sec'], 2)}s (lạnh) | {fmt_num(ps['single_paper_amortized_1ver_sec'], 2)}s (khấu hao lô)",
     "Thực nghiệm đơn bài trên Colab CPU đạt trung vị p50 = 3,08–4,82s (sát sàn rate-limit 3,0s của arXiv), p95 = 8,71–23,67s"),
    ("2. Thời gian trung bình mỗi bài báo trên tập yêu cầu",
     f"{fmt_num(ps['full_4shards_avg_sec_per_paper'], 4)}s/bài (4 phiên full) | {fmt_num(ps['all_5members_avg_sec_per_paper'], 4)}s/bài (đủ 25k bài)",
     f"Tổng wall-time 6 phiên = {fmt_num(ps['all_5members_wall_sec'], 1)}s ({fmt_num(ps['all_5members_wall_sec']/3600, 2)} giờ, tương đương {fmt_num(ps['all_5members_avg_sec_per_version'], 2)}s/version)"),
    ("3. Tổng thời gian khám phá danh mục (Entry Discovery)",
     f"{fmt_num(ps['discovery_4shards_sec'], 2)}s (4 shard full) | {fmt_num(ps['discovery_measured_total_sec'], 2)}s (+TRI resume)",
     f"{fmt_int(ps['discovery_4shards_calls'])} lô Atom (~0,105s/bài, chỉ chiếm 1,30% tổng wall-time) + {fmt_num(ps['id_boundary_tri_resume_sec'], 2)}s dò biên tháng"),
    ("4. Bộ nhớ RAM cực đại (Peak RSS)",
     f"{fmt_num(ps['peak_rss_mib'], 2)} MiB (OS ru_maxrss: {fmt_num(ps['os_reported_peak_rss_mib'], 2)} MiB)",
     "Chỉ chiếm 9,46% trên tổng 12.975,52 MiB RAM của máy ảo Google Colab CPU-only"),
    ("5. Bộ nhớ RAM trung bình gia quyền theo thời gian (Weighted Avg RSS)",
     f"{fmt_num(ps['weighted_avg_rss_mib'], 2)} MiB",
     f"Tính trên {fmt_int(ps['rss_total_samples_4shards'])} mẫu đo chu kỳ 0,5s (+{fmt_int(ps['rss_samples_tri_resume'])} mẫu phiên TRI-resume: 625,73 MiB)"),
    ("6. Dung lượng đĩa cực đại yêu cầu trên 1 phiên Colab (Peak Disk)",
     f"{fmt_num(ps['tri_resume_sampled_peak_disk_bytes']/1024**3, 2)} – {fmt_num(ps['max_member_end_disk_bytes']/1024**3, 2)} GiB",
     f"Đỉnh lấy mẫu TRI-resume: {fmt_int(ps['tri_resume_sampled_peak_disk_bytes'])} B; cuối phiên lớn nhất (TRIET): {fmt_int(ps['max_member_end_disk_bytes'])} B"),
    ("7. Dung lượng lưu trữ đầu ra cuối cùng (Final Output Size)",
     f"Corpus TeX/Bib sạch: {fmt_num(ps['final_clean_corpus_bytes']/1024**3, 2)} GiB | Course Parquet: {fmt_num(ps['final_course_export_total_bytes']/1048576, 2)} MiB",
     f"Working Parquet hợp nhất: {fmt_num(ps['final_merged_working_parquet_bytes']/1048576, 2)} MiB; papers (3,52 MiB) + references (74,23 MiB) = {fmt_int(ps['final_course_export_total_bytes'])} B"),
], columns=["Chỉ số hiệu năng bắt buộc (M1)", "Giá trị đo trên Colab CPU-only", "Giải thích & Ngữ cảnh kỹ thuật"])
display(style_table(df_7m, caption="Bảng 5.1. Tổng hợp đầy đủ 7 chỉ số hiệu năng bắt buộc của Crawler Milestone 1 trên Google Colab CPU-only."))

In [ ]:
sess_m1 = pd.DataFrame(m1["colab_sessions_full"])
view_s1 = pd.DataFrame({
    "Phiên chạy": sess_m1["session_label"],
    "Bài xử lý": sess_m1["processed_this_run"],
    "Hoàn tất lũy kế": sess_m1["completed_in_checkpoint"],
    "Thời gian (giờ)": sess_m1["runtime_sec"] / 3600.0,
    "TB giây/bài": sess_m1["avg_sec_per_processed"],
    "Peak RSS (MiB)": sess_m1["peak_rss_mib"],
    "Mean RSS (MiB)": sess_m1["avg_rss_mib"],
    "Đĩa cuối phiên (GiB)": sess_m1["end_disk_bytes"] / (1024**3),
    "Trạng thái dừng": sess_m1["stop_reason"],
})
display(style_table(
    view_s1,
    caption="Bảng 5.2. Chi tiết cả 6 phiên thực thi Google Colab CPU-only của 5 thành viên ở giai đoạn Milestone 1.",
    int_cols=["Bài xử lý", "Hoàn tất lũy kế"],
    bar_cols={"TB giây/bài": ("var(--ah-bar-fill)", 11.0)},
))

ph_raw = m1["phases_aggregated"]
df_ph = pd.DataFrame([
    {
        "Pha xử lý M1": k,
        "Số lượt gọi (calls)": int(v["calls"]),
        "Tổng giây cộng dồn": float(v["elapsed_sec"]),
        "Tổng giờ": float(v["elapsed_sec"]) / 3600.0,
        "TB giây / lượt gọi": float(v["elapsed_sec"]) / max(int(v["calls"]), 1),
    }
    for k, v in ph_raw.items()
]).sort_values("Tổng giây cộng dồn", ascending=False)
display(style_table(
    df_ph,
    caption="Bảng 5.3. Phân rã thời gian 12 pha crawler M1 trên 4 phiên full (tổng pha lớn hơn wall-time 1,115× nhờ chạy đa luồng gối đầu).",
    int_cols=["Số lượt gọi (calls)"],
    bar_cols={"Tổng giây cộng dồn": ("rgba(13, 148, 136, 0.22)", None)},
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.8, 4.1))
ax1.bar(sess_m1["session_label"], sess_m1["avg_sec_per_processed"], color=BOOK_PALETTE["navy"], width=0.58)
ax1.axhline(ps["all_5members_avg_sec_per_paper"], color=BOOK_PALETTE["crimson"], ls="--", lw=1.3, label=f"TB toàn nhóm ({fmt_num(ps['all_5members_avg_sec_per_paper'], 2)}s/bài)")
ax1.tick_params(axis="x", rotation=22); ax1.set_ylim(0, 11.5); ax1.grid(axis="y")
ax1.set_ylabel("Giây / bài xử lý"); ax1.set_title("Hình 5.1a. Tốc độ xử lý trung bình từng phiên M1")
ax1.legend(fontsize=8.2)
for idx_s, v_s in enumerate(sess_m1["avg_sec_per_processed"]):
    ax1.text(idx_s, v_s + 0.22, f"{fmt_num(v_s, 2)}s", ha="center", fontsize=8.0, fontweight="bold")

mem_s = sess_m1.dropna(subset=["peak_rss_mib"]).reset_index(drop=True)
xm = np.arange(len(mem_s)); wm = 0.35
ax2.bar(xm - wm/2, mem_s["avg_rss_mib"], wm, color=BOOK_PALETTE["teal"], label="Mean RSS (MiB)")
ax2.bar(xm + wm/2, mem_s["peak_rss_mib"], wm, color=BOOK_PALETTE["amber"], label="Peak RSS (MiB)")
ax2.set_xticks(xm); ax2.set_xticklabels(mem_s["session_label"], rotation=20)
ax2.set_ylim(0, 1480); ax2.grid(axis="y")
ax2.set_ylabel("Bộ nhớ RAM (MiB)"); ax2.set_title("Hình 5.1b. Mức tiêu thụ RAM trên Colab CPU (< 10% trần 12,7 GiB)")
ax2.legend(fontsize=8.2)
for idx_m, r_m in mem_s.iterrows():
    ax2.text(idx_m + wm/2, r_m["peak_rss_mib"] + 25, fmt_num(r_m["peak_rss_mib"], 0), ha="center", fontsize=7.9, fontweight="bold")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(10.2, 4.2))
ph_plot = df_ph.sort_values("Tổng giây cộng dồn", ascending=True)
cols_p = [BOOK_PALETTE["amber"] if p == "discovery" else BOOK_PALETTE["navy"] for p in ph_plot["Pha xử lý M1"]]
ax.barh(ph_plot["Pha xử lý M1"], ph_plot["Tổng giây cộng dồn"], color=cols_p, height=0.60)
ax.set_xscale("log"); ax.set_xlim(0.18, 6e5); ax.grid(axis="x")
ax.set_xlabel("Tổng giây cộng dồn (thang log10)")
ax.set_title("Hình 5.2. Phân bổ thời gian 12 pha crawler M1 (nổi bật Discovery = 2.094,9s & Tải nguồn khóa nhịp 3,0s)")
for idx_p, r_p in enumerate(ph_plot.to_dict("records")):
    ax.text(max(r_p["Tổng giây cộng dồn"], 0.22) * 1.18, idx_p,
            f"{fmt_num(r_p['Tổng giây cộng dồn'], 1)}s ({fmt_int(r_p['Số lượt gọi (calls)'])} calls)",
            va="center", fontsize=8.0, fontweight="bold")
plt.tight_layout(); plt.show()

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 1 — Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/profiler.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/profiler.py).
- Mã nguồn triển khai: [`src/crawler/benchmark_sessions.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/benchmark_sessions.py).
- Mã nguồn triển khai: [`tools/m1_colab_tri_resume.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m1_colab_tri_resume.py).